conda env: neurolens

# 01 — PDF Ingestion and Semantic Retrieval

This notebook builds the first working component of **NeuroLens-RAG**:

1. Read a scientific PDF.
2. Preserve page-level provenance.
3. Split the text into retrieval chunks.
4. Embed the chunks with a Sentence Transformer.
5. Retrieve relevant evidence for a natural-language question.
6. Evaluate retrieval with a small set of known-answer questions.
7. Save the index for later use in Streamlit and answer generation.

This version intentionally **does not call an LLM**. We first make retrieval measurable and trustworthy.

## Expected project layout

```text
neurolens-rag/
├── data/
│   └── papers/
│       └── journal.pcbi.1008943.pdf
├── artifacts/
├── notebooks/
│   └── 01_pdf_ingestion.ipynb
└── src/
```

Copy the paper into `data/papers/` before running the notebook.


In [1]:
from __future__ import annotations

import json
import re
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import pymupdf4llm
import torch
from sentence_transformers import SentenceTransformer

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

/Users/srinivasgovindasurampudi/miniconda3/envs/neurolens/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.15
PyTorch: 2.13.0
MPS available: True


## 1. Resolve project paths

The notebook may be launched from either the repository root or the `notebooks/` directory.  
The helper below searches upward for `environment.yml` or `.git`.


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root by searching parent directories."""
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Open the notebook from inside "
        "the neurolens-rag repository."
    )


PROJECT_ROOT = find_project_root()
PDF_PATH = PROJECT_ROOT / "data" / "papers" / "journal.pcbi.1008943.pdf"
INDEX_DIR = PROJECT_ROOT / "artifacts" / "paper_index"

INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PDF path:", PDF_PATH)
print("Index directory:", INDEX_DIR)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found at {PDF_PATH}\n"
        "Create data/papers/ and copy the paper there."
    )

Project root: /Users/srinivasgovindasurampudi/Projects/neurolens-rag
PDF path: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/data/papers/journal.pcbi.1008943.pdf
Index directory: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/artifacts/paper_index


## 2. Extract page-level Markdown

`page_chunks=True` returns one record per PDF page. Keeping the page number attached to every chunk will later allow us to cite evidence instead of producing untraceable answers.


In [3]:
page_records = pymupdf4llm.to_markdown(
    PDF_PATH,
    page_chunks=True,
)

print("Pages extracted:", len(page_records))
print("Keys in one page record:", page_records[0].keys())

page_preview = page_records[0]["text"][:1500]
print(page_preview)

Pages extracted: 25
Keys in one page record: dict_keys(['metadata', 'toc_items', 'page_boxes', 'text'])
PLOS COMPUTATIONAL BIOLOGY 

##### RESEARCH ARTICLE 

# Learning brain dynamics for decoding and predicting individual differences 

**Joyneel MisraID1‡*, Srinivas Govinda SurampudiID1‡, Manasij Venkatesh1, Chirag Limbachia**<sup>**2**</sup> **, Joseph JajaID1, Luiz PessoaID1,2*** 

**1** Department of Electrical and Computer Engineering, University of Maryland, College Park, Maryland, United States of America, **2** Department of Psychology and Maryland Neuroimaging Center, University of Maryland, College Park, Maryland, United States of America 

‡Authors with comparable contributions listed in random order. 

~~<u>a1111111111 a1111111111 a1111111111 a1111111111 a1111111111</u>~~ 



##### OPEN ACCESS 

**Citation:** Misra J, Surampudi SG, Venkatesh M, Limbachia C, Jaja J, Pessoa L (2021) Learning brain dynamics for decoding and predicting individual differences. PLoS Comput Biol 1

In [4]:
page_summary = pd.DataFrame(
    {
        "page": [record["metadata"]["page_number"] for record in page_records],
        "characters": [len(record["text"]) for record in page_records],
        "words": [len(record["text"].split()) for record in page_records],
    }
)

page_summary.head()

,page,characters,words
0,1,4251,557
1,2,4692,642
2,3,3719,525
3,4,1950,266
4,5,4442,653


## 3. Clean and chunk the extracted text

A retrieval chunk should be:

- small enough to focus on one idea;
- large enough to preserve scientific context;
- traceable to a page;
- overlapping with adjacent chunks so a sentence near a boundary is not lost.

For the prototype, we use a simple word-window chunker. Later we can compare it against section-aware and sentence-aware chunking.


In [5]:
@dataclass(frozen=True)
class TextChunk:
    chunk_id: str
    page: int
    text: str
    start_word: int
    end_word: int
    source_file: str


def clean_markdown(text: str) -> str:
    """Apply conservative cleanup while preserving scientific content."""
    text = text.replace("\u00ad", "")  # soft hyphen
    text = re.sub(r"-\s*\n\s*", "", text)  # join line-broken words
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def chunk_words(
    text: str,
    *,
    page: int,
    source_file: str,
    chunk_size: int = 220,
    overlap: int = 50,
) -> list[TextChunk]:
    """Split one page into overlapping word windows."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must satisfy 0 <= overlap < chunk_size.")

    words = text.split()
    if not words:
        return []

    step = chunk_size - overlap
    chunks: list[TextChunk] = []

    for chunk_index, start in enumerate(range(0, len(words), step)):
        end = min(start + chunk_size, len(words))
        chunk_text = " ".join(words[start:end]).strip()

        if len(chunk_text) < 80:
            continue

        chunks.append(
            TextChunk(
                chunk_id=f"page-{page:03d}-chunk-{chunk_index:03d}",
                page=page,
                text=chunk_text,
                start_word=start,
                end_word=end,
                source_file=source_file,
            )
        )

        if end == len(words):
            break

    return chunks

In [6]:
chunks: list[TextChunk] = []

for record in page_records:
    page_number = int(record["metadata"]["page_number"])
    cleaned_text = clean_markdown(record["text"])

    chunks.extend(
        chunk_words(
            cleaned_text,
            page=page_number,
            source_file=PDF_PATH.name,
            chunk_size=220,
            overlap=50,
        )
    )

print("Total chunks:", len(chunks))

chunks_df = pd.DataFrame([asdict(chunk) for chunk in chunks])
chunks_df[["chunk_id", "page", "start_word", "end_word", "text"]].head()

Total chunks: 76


,chunk_id,page,start_word,end_word,text
0,page-001-chunk-000,1,0,220,PLOS COMPUTATIONAL BIOLOGY ##### RESEARCH ARTI...
1,page-001-chunk-001,1,170,390,available here: https://doi.org/10.1371/journa...
2,page-001-chunk-002,1,340,557,( 60%) at the level of voxels. The model was a...
3,page-002-chunk-000,2,0,220,PLOS COMPUTATIONAL BIOLOGY Learning brain dyna...
4,page-002-chunk-001,2,170,390,pattern analysis. We believe our approach prov...


In [7]:
print(chunks[0].chunk_id)
print("Page:", chunks[0].page)
print(chunks[0].text[:1200])

page-001-chunk-000
Page: 1
PLOS COMPUTATIONAL BIOLOGY ##### RESEARCH ARTICLE # Learning brain dynamics for decoding and predicting individual differences **Joyneel MisraID1‡*, Srinivas Govinda SurampudiID1‡, Manasij Venkatesh1, Chirag Limbachia**<sup>**2**</sup> **, Joseph JajaID1, Luiz PessoaID1,2*** **1** Department of Electrical and Computer Engineering, University of Maryland, College Park, Maryland, United States of America, **2** Department of Psychology and Maryland Neuroimaging Center, University of Maryland, College Park, Maryland, United States of America ‡Authors with comparable contributions listed in random order. ~~<u>a1111111111 a1111111111 a1111111111 a1111111111 a1111111111</u>~~ ##### OPEN ACCESS **Citation:** Misra J, Surampudi SG, Venkatesh M, Limbachia C, Jaja J, Pessoa L (2021) Learning brain dynamics for decoding and predicting individual differences. PLoS Comput Biol 17(9): e1008943. https://doi.org/10.1371/journal. pcbi.1008943 **Editor:** Daniele Marinazzo, Gh

## 4. Create document embeddings

`all-MiniLM-L6-v2` is small enough for a laptop prototype. We normalize embeddings so that a matrix dot product is equivalent to cosine similarity.

For a 25-page paper, CPU inference is already fast and avoids device-specific surprises. We can benchmark MPS later.


In [8]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DEVICE = "cpu"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=EMBEDDING_DEVICE,
)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding device:", embedding_model.device)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5742.72it/s]


Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding device: cpu
Embedding dimension: 384


/var/folders/r4/spznqqj55yjg7yv8f_ym1pmm0000gn/T/ipykernel_27547/3388133714.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [9]:
chunk_texts = [chunk.text for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

chunk_embeddings = np.asarray(chunk_embeddings, dtype=np.float32)

print("Embedding matrix shape:", chunk_embeddings.shape)
print(
    "Mean vector norm:",
    np.linalg.norm(chunk_embeddings, axis=1).mean().round(4),
)

Batches: 100%|██████████| 3/3 [00:00<00:00,  3.01it/s]

Embedding matrix shape: (76, 384)
Mean vector norm: 1.0


## 5. Implement semantic retrieval

Because the corpus is tiny, a NumPy matrix multiplication is preferable to adding a vector database. The output includes page numbers and chunk identifiers for provenance.


In [10]:
def retrieve_chunks(
    query: str,
    *,
    model: SentenceTransformer,
    embeddings: np.ndarray,
    chunks: list[TextChunk],
    top_k: int = 5,
) -> pd.DataFrame:
    """Return the top semantic matches for a natural-language query."""
    query = query.strip()
    if not query:
        raise ValueError("query must not be empty.")
    if top_k <= 0:
        raise ValueError("top_k must be positive.")
    if len(chunks) != len(embeddings):
        raise ValueError("chunks and embeddings must have the same length.")

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0].astype(np.float32)

    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][: min(top_k, len(chunks))]

    rows = []
    for rank, index in enumerate(top_indices, start=1):
        chunk = chunks[int(index)]
        rows.append(
            {
                "rank": rank,
                "score": float(scores[index]),
                "page": chunk.page,
                "chunk_id": chunk.chunk_id,
                "text": chunk.text,
            }
        )

    return pd.DataFrame(rows)

In [11]:
query = "How were training and test participants separated for movie classification?"

results = retrieve_chunks(
    query,
    model=embedding_model,
    embeddings=chunk_embeddings,
    chunks=chunks,
    top_k=5,
)

results[["rank", "score", "page", "chunk_id", "text"]]

,rank,score,page,chunk_id,text
0,1,0.386715,14,page-014-chunk-002,"for training, and the remaining 76 participant..."
1,2,0.376124,5,page-005-chunk-001,915 parameters) approximated that of the GRU c...
2,3,0.373934,13,page-013-chunk-002,"trained data (see also [73]). Finally, we stre..."
3,4,0.367174,3,page-003-chunk-002,below were obtained with data not used in any ...
4,5,0.362276,15,page-015-chunk-001,vector of class scores: ^ **_y_** _t_ ¼ Softma...


## 6. Inspect several domain-specific queries

A retrieval system should be tested against questions whose answers occur in different parts of the paper.


In [12]:
example_queries = [
    "What happened to clip classification when temporal order was shuffled?",
    "How many latent dimensions retained nearly full classification accuracy?",
    "How was saliency calculated for individual brain regions?",
    "Which brain networks caused the largest accuracy decrease when lesioned?",
    "How were fluid intelligence and verbal IQ predicted?",
]

for example_query in example_queries:
    print("\n" + "=" * 100)
    print("QUERY:", example_query)

    display(
        retrieve_chunks(
            example_query,
            model=embedding_model,
            embeddings=chunk_embeddings,
            chunks=chunks,
            top_k=3,
        )[["rank", "score", "page", "text"]]
    )


QUERY: What happened to clip classification when temporal order was shuffled?


,rank,score,page,text
0,1,0.516846,3,below were obtained with data not used in any ...
1,2,0.461257,5,915 parameters) approximated that of the GRU c...
2,3,0.437246,6,the variability of participant trajectories ar...



QUERY: How many latent dimensions retained nearly full classification accuracy?


,rank,score,page,text
0,1,0.548088,6,", which yielded very low prediction accuracy, ..."
1,2,0.524282,6,the variability of participant trajectories ar...
2,3,0.469279,5,915 parameters) approximated that of the GRU c...



QUERY: How was saliency calculated for individual brain regions?


,rank,score,page,text
0,1,0.733059,7,Video). The saliency time series in a few brai...
1,2,0.679050,9,PLOS COMPUTATIONAL BIOLOGY Learning brain dyna...
2,3,0.627364,18,PLOS COMPUTATIONAL BIOLOGY Learning brain dyna...



QUERY: Which brain networks caused the largest accuracy decrease when lesioned?


,rank,score,page,text
0,1,0.521985,8,PLOS COMPUTATIONAL BIOLOGY Learning brain dyna...
1,2,0.510953,9,PLOS COMPUTATIONAL BIOLOGY Learning brain dyna...
2,3,0.490901,1,PLOS COMPUTATIONAL BIOLOGY ##### RESEARCH ARTI...



QUERY: How were fluid intelligence and verbal IQ predicted?


,rank,score,page,text
0,1,0.561506,12,"can be further studied, for instance, by devel..."
1,2,0.554094,9,idiosyncratic to a particular clip. Fig 6 show...
2,3,0.514117,20,is needed because a single test is applied to ...


## 7. Add a small retrieval benchmark

We define expected page sets from our knowledge of this paper. The score answers a simple question:

> Did at least one of the top-*k* retrieved chunks come from a page containing the expected answer?

This is not a complete RAG evaluation, but it prevents us from judging retrieval only by appearance.


In [13]:
retrieval_benchmark = [
    {
        "query": "How many participants were used for movie training and testing?",
        "expected_pages": {14},
    },
    {
        "query": "What was the effect of temporal shuffling on clip classification?",
        "expected_pages": {3, 4, 5},
    },
    {
        "query": "How well did low-dimensional trajectories classify movie clips?",
        "expected_pages": {5, 6},
    },
    {
        "query": "How did the authors define gradient saliency?",
        "expected_pages": {17, 18},
    },
    {
        "query": "What did the network lesion experiments reveal?",
        "expected_pages": {7, 8, 9},
    },
]


def evaluate_retrieval_hit_rate(
    benchmark: Iterable[dict],
    *,
    top_k: int,
) -> pd.DataFrame:
    rows = []

    for item in benchmark:
        result = retrieve_chunks(
            item["query"],
            model=embedding_model,
            embeddings=chunk_embeddings,
            chunks=chunks,
            top_k=top_k,
        )

        retrieved_pages = set(result["page"].astype(int))
        expected_pages = set(item["expected_pages"])
        hit = bool(retrieved_pages & expected_pages)

        rows.append(
            {
                "query": item["query"],
                "expected_pages": sorted(expected_pages),
                "retrieved_pages": sorted(retrieved_pages),
                f"hit@{top_k}": hit,
            }
        )

    return pd.DataFrame(rows)


benchmark_results = evaluate_retrieval_hit_rate(
    retrieval_benchmark,
    top_k=5,
)

benchmark_results

,query,expected_pages,retrieved_pages,hit@5
0,How many participants were used for movie trai...,[14],"[7, 13, 14, 20, 21]",True
1,What was the effect of temporal shuffling on c...,"[3, 4, 5]","[3, 5, 6, 11, 12]",True
2,How well did low-dimensional trajectories clas...,"[5, 6]","[5, 6, 11, 13]",True
3,How did the authors define gradient saliency?,"[17, 18]","[7, 9, 17, 18]",True
4,What did the network lesion experiments reveal?,"[7, 8, 9]","[7, 8, 18, 22, 24]",True


In [14]:
hit_column = "hit@5"
hit_rate = benchmark_results[hit_column].mean()

print(f"{hit_column}: {hit_rate:.1%}")

hit@5: 100.0%


## 8. Save the local paper index

The generated files belong in `artifacts/`, which should remain outside Git. They can be regenerated from the PDF and notebook.

Saved outputs:

- `chunks.jsonl`: text and provenance metadata;
- `embeddings.npy`: normalized dense vectors;
- `index_metadata.json`: model and chunking configuration.


In [15]:
chunks_path = INDEX_DIR / "chunks.jsonl"
embeddings_path = INDEX_DIR / "embeddings.npy"
metadata_path = INDEX_DIR / "index_metadata.json"

with chunks_path.open("w", encoding="utf-8") as file:
    for chunk in chunks:
        file.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")

np.save(embeddings_path, chunk_embeddings)

index_metadata = {
    "source_pdf": PDF_PATH.name,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": int(chunk_embeddings.shape[1]),
    "number_of_pages": len(page_records),
    "number_of_chunks": len(chunks),
    "chunk_size_words": 220,
    "overlap_words": 50,
    "normalized_embeddings": True,
}

metadata_path.write_text(
    json.dumps(index_metadata, indent=2),
    encoding="utf-8",
)

print("Saved:", chunks_path)
print("Saved:", embeddings_path)
print("Saved:", metadata_path)

Saved: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/artifacts/paper_index/chunks.jsonl
Saved: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/artifacts/paper_index/embeddings.npy
Saved: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/artifacts/paper_index/index_metadata.json


## 9. Verify that the saved index can be reloaded

A reusable index must not depend on notebook memory.


In [16]:
reloaded_embeddings = np.load(embeddings_path)

with chunks_path.open("r", encoding="utf-8") as file:
    reloaded_chunks = [
        TextChunk(**json.loads(line))
        for line in file
    ]

reloaded_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

assert reloaded_embeddings.shape == chunk_embeddings.shape
assert len(reloaded_chunks) == len(chunks)
assert reloaded_metadata["number_of_chunks"] == len(chunks)

print("Reload verification passed.")
print(reloaded_metadata)

Reload verification passed.
{'source_pdf': 'journal.pcbi.1008943.pdf', 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dimension': 384, 'number_of_pages': 25, 'number_of_chunks': 76, 'chunk_size_words': 220, 'overlap_words': 50, 'normalized_embeddings': True}


## 10. Exercises and next refactor

Before adding an LLM, inspect failures and try:

1. `chunk_size` values of 120, 220, and 350 words.
2. Overlaps of 20, 50, and 80 words.
3. A section-aware chunker that preserves headings.
4. A scientific embedding model.
5. A lexical BM25 baseline.
6. Hybrid dense + lexical retrieval.
7. A reranker over the top retrieved chunks.

Once this notebook is reliable, move reusable code into:

```text
src/neurolens/rag/
├── __init__.py
├── ingestion.py
├── chunking.py
├── embeddings.py
└── retrieval.py
```

The next notebook should be:

```text
02_pdf_rag_answering.ipynb
```

That notebook will add answer generation, but only after the retriever returns explicit evidence with page-level provenance.
